# Entrenamiento de detección de emociones con CNN

Importación de paquetes

In [1]:
import os

import cv2
import numpy as np
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical

Variables del dataset

In [2]:
DATASET_DIR = "dataset_processed"
IMG_SIZE = 32
MODEL_PATH = "model_results/cnn_emotion.keras"

CLASSES = sorted(os.listdir(os.path.join(DATASET_DIR, "train")))
print("Clases:", CLASSES)

Clases: ['Angry', 'Fear', 'Happy', 'Sad', 'Suprise']


Cargar imágenes en gris y 32x32

In [3]:
def load_split(split_name):
    X, y = [], []
    split_dir = os.path.join(DATASET_DIR, split_name)
    for label_idx, label in enumerate(CLASSES):
        class_dir = os.path.join(split_dir, label)
        for fname in sorted(os.listdir(class_dir)):
            img = cv2.imread(os.path.join(class_dir, fname), cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X.append(img)
            y.append(label_idx)
    X = np.array(X, dtype="float32") / 255.0
    X = X.reshape(-1, IMG_SIZE, IMG_SIZE, 1)
    y = to_categorical(y, num_classes=len(CLASSES))
    return X, y


X_train, y_train = load_split("train")
X_test, y_test = load_split("test")

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (39100, 32, 32, 1) Test: (9825, 32, 32, 1)


Definición de la CNN

In [4]:
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(len(CLASSES), activation="softmax"),
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 32, 32, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 16, 16, 32)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 16, 16, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 8, 8, 64)         0         
 2D)                                                             
                                                                 
 flatten (Flatten)           (None, 4096)              0         
                                                                 
 dense (Dense)               (None, 128)               5

Entrenamiento

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=30,
    batch_size=64,
)

test_loss, test_accuracy = model.evaluate(X_test, y_test)
print("Accuracy en test:", test_accuracy)

Epoch 1/20
550/550 [==============================] - 7s 8ms/step - loss: 1.3062 - accuracy: 0.4471 - val_loss: 2.4052 - val_accuracy: 0.0054
Epoch 2/20
550/550 [==============================] - 4s 7ms/step - loss: 1.1485 - accuracy: 0.5238 - val_loss: 2.2462 - val_accuracy: 0.2036
Epoch 3/20
550/550 [==============================] - 3s 6ms/step - loss: 1.0703 - accuracy: 0.5601 - val_loss: 1.7154 - val_accuracy: 0.4123
Epoch 4/20
550/550 [==============================] - 3s 6ms/step - loss: 1.0058 - accuracy: 0.5887 - val_loss: 1.8423 - val_accuracy: 0.3604
Epoch 5/20
550/550 [==============================] - 3s 6ms/step - loss: 0.9451 - accuracy: 0.6161 - val_loss: 1.5235 - val_accuracy: 0.4634
Epoch 6/20
550/550 [==============================] - 3s 6ms/step - loss: 0.9037 - accuracy: 0.6377 - val_loss: 2.0000 - val_accuracy: 0.3404
Epoch 7/20
550/550 [==============================] - 3s 6ms/step - loss: 0.8490 - accuracy: 0.6610 - val_loss: 2.2663 - val_accuracy: 0.3294
Epoch 

Guardar el modelo entrenado

In [6]:
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
model.save(MODEL_PATH)
print(f"Modelo guardado en {MODEL_PATH}")

Modelo guardado en model_results/cnn_emotion.keras
